# Operation evidence by category and valence

Complementary to `valence_evidence_by_operation.ipynb` (the category
classifier version in this same directory) -- same structure and helper
functions, with category and operation swapping roles: here `regressor_label`
is this classifier's own trained target, one of maintain/suppress/switch/
clear, so each row's own **self-evidence** (`evidence_<its own true
operation>`) is the value plotted throughout -- "how much did the classifier
believe this trial's operation was its own true operation". Category
(face/place) becomes the secondary, trial_type-derived grouping dimension
instead.

No abstraction layer beyond a handful of small helper functions -- every
plot below is a short, self-contained function you can copy and adapt.

The subtraction-against-baseline plot from the category-classifier notebook
(operation minus maintain) is left out here -- operation is this
classifier's own directly-decoded target, not a secondary factor to
baseline against, so it doesn't apply.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # run from _interactive_notebooks/, repo root one level up
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from workflows.generate_report import (
    resolve_desc, list_subject_dirs, parse_subjects_arg, subject_paths, broadcast_trial_label,
    resolve_marker_label, draw_event_annotations,
)
from utils.mvpa_common import is_excluded_trial_type, is_trial_end_marker
from _interactive_notebooks.decoding_results_toolkit import (
    select_evidence_value, aggregate_by_subject_window, bin_by_size, compare_conditions_by_bin,
    annotate_significance, max_reliable_window_index,
)

%matplotlib inline
sns.set_theme(style="ticks")

from matplotlib.lines import Line2D

def add_shared_legend(fig, color_map, title=None, ncol=None):
    """One legend for the whole figure, placed below every subplot --
    built directly from color_map ({label: color}) rather than pulled
    from any one subplot's own handles, so it's correct regardless of
    which subplots happen to have data for every hue level."""
    handles = [Line2D([0], [0], color=color, lw=2, label=label) for label, color in color_map.items()]
    fig.legend(handles=handles, loc="lower center", ncol=ncol or len(color_map),
               title=title, bbox_to_anchor=(0.5, -0.02), fontsize=9)

## Configuration -- edit these

In [ ]:
# -- where the data is -- two options:
# (1) a single already-merged CSV (e.g. generate_report.py's own
#     <desc>_group_decoding_results.csv, or any decoding_results.csv) --
#     set GROUP_CSV_PATH and leave the rest alone.
# (2) load + concatenate each subject's own decoding_results.csv straight
#     from the cluster's output tree -- leave GROUP_CSV_PATH = None and set
#     ANALYSIS_OUTPUT_DIR/DESC/SUBJECTS instead.
GROUP_CSV_PATH = None  # e.g. "/path/to/vvps_operation_timecourse_classifier_group_decoding_results.csv"

ANALYSIS_OUTPUT_DIR = "/path/to/analysis/output"
DESC = "vvps_operation_timecourse_classifier"  # (!) placeholder -- set to the real operation classifier's model_descr
CONFIG_PATH = None      # optional: read DESC from a config's model.desc instead
SUBJECTS = None         # comma-separated list, a subject-list.txt path, or None for everyone

# per-category colors -- distinct from valence's red/blue below so the two
# hue dimensions are never visually confused
CATEGORY_COLORS = {"face": "purple", "place": "teal"}
VALENCE_COLORS = {"pos": "red", "neg": "blue"}

# fixed display order for this classifier's own trained categories
# (maintain/suppress/switch/clear) -- just ordering, no color meaning here,
# since operation is the panel dimension below, not a hue
OPERATION_ORDER = ["maintain", "switch", "suppress", "clear"]

TR_SECONDS = 0.460     # seconds per TR -- x-axis everywhere below is window_index * TR_SECONDS

# friendly name for a group of mutually-exclusive real trial_type values,
# for the event markers computed below (see compute_marker_positions) --
# same shape/values as this classifier's own
# model_conditions.timecourse_decoding.annotation_labels; a position not
# covered by any group here just falls back to auto-joining its own
# distinct values instead
ANNOTATION_LABELS = {
    "view": ["view_face", "view_place"],
    "operation": [
        "maintain_face", "suppress_face", "switch_face", "clear_face",
        "maintain_place", "suppress_place", "switch_place", "clear_place",
    ],
}

# generic y-axis label for all plots below.
Y_LABEL = "vvps operation classifier evidence\ntrained on working memory task"

OUTPUT_DIR = "./operation_evidence_analysis"

# -- stats: pos vs. neg (within category) --
BIN_SIZE = 3                              # TRs per non-overlapping stat window
STAT_METHOD = "ttest_1samp_diff"          # see decoding_results_toolkit.STAT_METHODS
POS_VS_NEG_ALTERNATIVE = "two-sided"
ALPHA = 0.05

# a trial's real length varies (jittered ITI before the next trial's own
# anchor) -- only a shrinking handful of unusually long trials still have
# data at the far tail of the window, so the x-axis is truncated once
# subject coverage for any (operation, category, valence) combination
# drops below this fraction of the full group, rather than plotting a
# handful of outlier trials as if they were a real group average.
MIN_SUBJECT_FRACTION = 0.9

# how many extra TRs to extend each trial's own window past that reliable
# cutoff -- by absolute volume_of_interest from the trial's own start, not by
# filtering the natural window_index column, so a short trial's extension
# keeps going even once it's crossed into what's officially the next trial
# (borrowing its fixation tail and/or the next trial's own view period,
# whichever comes next -- see the load cell below). Deliberately
# double-dipping a frame into two trials' own windows there, rather than
# cutting off cleanly at whatever trial boundary happens to be nearby.
# Set to 0 to stop exactly at the reliable cutoff, no borrowing.
EXTRA_TRAILING_TRS = 6

# how many extra TRs to keep past that reliable cutoff -- shows the trial's
# fixation tail transitioning into the next trial's own view period, at the
# cost of a noisier, more sparsely-supported group average out there (by
# design: the whole point is to see that mixed-content tail, not to treat it
# as a reliable estimate). Set to 0 to truncate exactly at the reliable cutoff.
EXTRA_TRAILING_TRS = 6

## Load data + add a few columns for easy filtering

In [ ]:
if GROUP_CSV_PATH:
    df = pd.read_csv(GROUP_CSV_PATH, dtype={"subject": str}, keep_default_na=True, na_values=[""])
    desc = DESC
    print(f"Loaded {len(df)} rows from {GROUP_CSV_PATH} -- {df['subject'].nunique()} subject(s)")
else:
    desc = resolve_desc(None, CONFIG_PATH) if CONFIG_PATH else DESC
    subjects_arg = parse_subjects_arg(SUBJECTS) if SUBJECTS else None
    subjects = list_subject_dirs(ANALYSIS_OUTPUT_DIR, desc, subjects=subjects_arg)
    frames = []
    for s in subjects:
        path = subject_paths(ANALYSIS_OUTPUT_DIR, desc, s)["decoding_raw"]
        if os.path.exists(path):
            frames.append(pd.read_csv(path, dtype={"subject": str}, keep_default_na=True, na_values=[""]))
    df = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(df)} rows from {len(subjects)} subject(s)")

# continuous timecourse decoding leaves regressor_label blank on real per-TR
# rows that aren't a scored operation -- e.g. view_face/probe/fixation
# frames (see README.md section 4). keep_default_na/na_values above make
# sure a blank CSV field reads back as a real NaN rather than relying on
# pandas' default silently; replace() is a second, cheap safety net
# catching any literal empty string that slips through either read.
df = df.replace(r"^\s*$", pd.NA, regex=True)

# regressor_label is only ever set on the specific real event a classifier's
# `conditions` targets (the operate period -- maintain_face/suppress_place/
# ..., not the earlier view period or the trailing fixation) --
# broadcasting it across every row of its own trial (boldfile, trial_index)
# is what lets "which operation was this trial" stay defined throughout the
# whole trial, so evidence during the view period or fixation tail can
# still be asked "evidence for the operation this trial actually was"
# instead of going blank outside the operate cue itself (same mechanism
# generate_report.py's timecourse page already relies on -- see
# broadcast_trial_label).
df["regressor_label"] = broadcast_trial_label(df, "regressor_label")
df["operation"] = df["regressor_label"]                                    # maintain / suppress / switch / clear -- NaN on unscored frames

# -- a few extra columns to make filtering/sorting simple --
df["valence"] = df["task"].map({"WMpos": "pos", "WMneg": "neg"})           # pos / neg

# category, unlike regressor_label above, is literally present on every
# view/operate frame of a trial ("view_face"/"maintain_face" both carry
# "face") but goes blank on the trailing fixation tail -- every downstream
# groupby/filter keyed on "category" (Plot 1, below) would otherwise
# silently drop that tail, since pandas groupby drops NaN keys by default.
# Broadcasting fixes this the same way as regressor_label above: a trial
# has exactly one category for its whole duration.
df["category"] = df["trial_type"].str.extract(r"(face|place)", expand=False)
df["category"] = broadcast_trial_label(df, "category")

# self-evidence using each row's OWN natural trial -- only to find the
# reliable per-trial cutoff below; the actually-plotted "value" gets
# recomputed per the OWNING trial once the window is extended, since a
# borrowed tail frame's evidence must be judged against the trial it was
# borrowed FROM, not whatever trial that frame's own content belongs to.
# select_evidence_value (vectorized numpy indexing, not a row-wise .apply())
# drops rows whose regressor_label doesn't match any evidence_ column
# (an unqualified/excluded trial) rather than keeping them with a
# manually-built NaN, which pandas can silently store as *object* dtype and
# crash scipy.stats.ttest_1samp later.
natural = select_evidence_value(df, value="self", new_col="value")

max_window = max_reliable_window_index(natural, group_cols=["operation", "category", "valence"],
                                        min_fraction=MIN_SUBJECT_FRACTION)
n_dropped = (natural["window_index"] > max_window).sum()
print(f"reliable per-trial coverage extends to window_index <= {max_window} "
      f"({n_dropped} row(s) beyond it too sparse under each trial's own natural length)")
WINDOW_LEN = max_window + 1 + EXTRA_TRAILING_TRS

# real trial starts only: window_index == 0 AND a genuine anchor match --
# regressor_label is never set on a trial an end_cut started (see
# label_rows_optional/build_timecourse_instructions/partition_into_trials),
# only a real trial_start_event anchor gets its own extended window here.
trial_starts = df.loc[(df["window_index"] == 0) & df["regressor_label"].notna(),
                       ["boldfile", "trial_index", "volume_of_interest", "operation", "category", "valence"]].copy()
trial_starts = trial_starts.rename(columns={
    "trial_index": "owner_trial_index", "volume_of_interest": "start_vol",
    "operation": "owner_operation", "category": "owner_category", "valence": "owner_valence",
})

# for each real trial, pull every row within WINDOW_LEN TRs of ITS OWN start
# -- by absolute volume_of_interest, not the natural window_index column, so
# a short trial's window keeps counting even once volume_of_interest has
# moved into what's officially the next trial (or a block-boundary rest
# period). One exact-match merge per offset (not a full cross join) keeps
# this cheap: output size is n_real_trials * WINDOW_LEN, not
# n_real_trials * len(df); a frame two trials' windows both reach lands in
# the output twice, once per owner -- the intended double-dip.
frames = []
for offset in range(WINDOW_LEN):
    probe = trial_starts.assign(volume_of_interest=trial_starts["start_vol"] + offset)
    hit = df.merge(probe, on=["boldfile", "volume_of_interest"])
    hit["window_index"] = offset
    frames.append(hit)
df = pd.concat(frames, ignore_index=True)

# every plotted row is now keyed on the OWNING trial's own true operation --
# not necessarily this row's own natural trial, for a borrowed tail frame --
# so a borrowed frame's evidence_<> lookup reflects "how much did the
# classifier still believe this was the ORIGINATING trial's operation",
# not whatever trial the frame's own real content actually belongs to.
df["trial_index"] = df["owner_trial_index"]
df["regressor_label"] = df["owner_operation"]
df["operation"] = df["owner_operation"]
df["category"] = df["owner_category"]
df["valence"] = df["owner_valence"]
df = select_evidence_value(df, value="self", new_col="value")

df["time_sec"] = df["window_index"] * TR_SECONDS                           # x-axis for every plot below

OPERATIONS = [op for op in OPERATION_ORDER if op in df["operation"].unique()]   # whichever operations this classifier actually decodes
CATEGORIES = [c for c in CATEGORY_COLORS if c in df["category"].unique()]
print("operations found:", OPERATIONS)
print("categories found:", CATEGORIES)

df[["subject", "operation", "category", "valence", "time_sec", "value"]].head()

## Event markers -- where the view/operate/fixation periods actually fall

Same idea as `generate_report.py`'s own timecourse annotations (`compute_event_markers`/`draw_event_annotations`), computed directly from this dataframe's own real (un-broadcast) `trial_type` and `window_index` -- no `master_spreadsheet.csv` or config file needed, since `window_index` here is already relative to trial start (the same anchor the report page uses) and `trial_type` is already each frame's real, per-frame value. A dotted line marks each ordinal position (1st real sub-event in the trial, 2nd, ...) seen in most trials; `ANNOTATION_LABELS` above names a group of mutually-exclusive values at a position (e.g. one of 8 operation x category values) with one friendly label instead of auto-joining them.

In [ ]:
def compute_marker_positions(data, annotation_labels=None, min_frequency=0.5, max_label_values=4):
    events = data[["boldfile", "trial_index", "window_index", "trial_type"]].drop_duplicates()
    # a frame with no real event covering it (a brief, genuine gap in the
    # dense per-volume table -- e.g. between one event ending and the next
    # starting) must not count as its own ordinal position, or it shifts
    # every later position by one for whichever trials happen to have that
    # gap -- same reason compute_event_markers drops NaN event_index rows
    # before counting positions
    events = events[events["trial_type"].notna()]
    events = events.sort_values(["boldfile", "trial_index", "window_index"])

    # merge consecutive same-trial_type rows within a trial into one
    # sub-event -- mirrors compute_event_markers' own merge step (a
    # repeated run of the same real value is one occurrence, not one
    # per TR)
    same_trial = (events["boldfile"] == events["boldfile"].shift()) & (events["trial_index"] == events["trial_index"].shift())
    same_type = events["trial_type"] == events["trial_type"].shift()
    run_id = (~(same_trial & same_type)).cumsum()
    merged = events.groupby(run_id, sort=False).agg(
        boldfile=("boldfile", "first"), trial_index=("trial_index", "first"),
        trial_type=("trial_type", "first"), start=("window_index", "min"),
    ).reset_index(drop=True)

    merged["position"] = merged.groupby(["boldfile", "trial_index"]).cumcount() + 1

    # a trial_index whose own first frame is itself a block-boundary marker
    # (EndFixation/start_block/end_block) was created by partition_into_trials'
    # end_cut resetting the window, not by a real trial anchor -- it's
    # administrative filler, not a trial, and must not contribute its own
    # "position 1" here (mixing with a real trial's own position 1 and
    # corrupting resolve_marker_label's result there)
    admin_starts = merged[(merged["position"] == 1) & merged["trial_type"].apply(is_trial_end_marker)]
    admin_keys = set(admin_starts[["boldfile", "trial_index"]].itertuples(index=False, name=None))
    if admin_keys:
        merged = merged[~merged[["boldfile", "trial_index"]].apply(tuple, axis=1).isin(admin_keys)]

    n_trials = merged[["boldfile", "trial_index"]].drop_duplicates().shape[0]

    markers = []
    for position, group in merged.groupby("position"):
        frequency = group[["boldfile", "trial_index"]].drop_duplicates().shape[0] / n_trials
        if frequency < min_frequency:
            continue
        distinct_types = sorted(group["trial_type"].dropna().unique())
        # a "_nan"/"nan_..." trial_type (a malformed trial with a missing
        # item) is noise, not a real value worth covering or displaying
        real_types = [t for t in distinct_types if "nan" not in str(t).lower()] or distinct_types
        # a position that's only ever administrative/excluded content
        # (fixation/trial_fixation/EndFixation/...) is a rest/gap marker,
        # not a condition worth naming
        if all(is_excluded_trial_type(t) for t in real_types):
            label = ""
        else:
            label = resolve_marker_label(real_types, annotation_labels, max_label_values)
        markers.append({
            "trial_type": label,
            "mean_start": float(group["start"].mean()),
            "std_start": float(group["start"].std(ddof=0)) if len(group) > 1 else 0.0,
        })
    return sorted(markers, key=lambda m: m["mean_start"])


EVENT_MARKERS = compute_marker_positions(df, annotation_labels=ANNOTATION_LABELS)
EVENT_MARKERS

## Plot 1: operation evidence by category, one figure per valence

Subplots -- one per operation found above -- each overlaying category
(`CATEGORY_COLORS`) -- separate figures for pos and neg trials.

In [ ]:
def plot_operations_by_category(data, title):
    # one row per (subject, operation, category, time) -- so seaborn's CI
    # band reflects between-subject variability, not raw single-trial noise
    per_subject = data.groupby(["subject", "operation", "category", "time_sec"])["value"].mean().reset_index()

    fig, axes = plt.subplots(1, len(OPERATIONS), figsize=(5 * len(OPERATIONS), 4.5), sharex=True, sharey=True)
    for ax, op in zip(axes, OPERATIONS):
        sns.lineplot(data=per_subject[per_subject["operation"] == op], x="time_sec", y="value",
                     hue="category", hue_order=CATEGORIES, palette=CATEGORY_COLORS, ax=ax, legend=False)
        draw_event_annotations(ax, EVENT_MARKERS, tr=TR_SECONDS, show_labels=True)
        ax.set_title(op)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel(Y_LABEL)
    add_shared_legend(fig, CATEGORY_COLORS, title="category")
    fig.suptitle(title, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0.08, 1, 0.93])
    return fig


for valence in ("pos", "neg"):
    plot_operations_by_category(df[df["valence"] == valence], f"{desc}: operation evidence by category -- {valence}")
    plt.show()

## Statistics: pos vs. neg (within category)

Non-overlapping `BIN_SIZE`-TR windows; `STAT_METHOD` (default a one-sample t-test on each subject's own paired difference score) run per bin, per category. Collapsed across operation (maintain/switch/suppress/clear pooled -- self-evidence already normalizes each trial to its own true operation, so pooling is meaningful) -- see `_interactive_notebooks/decoding_results_toolkit.py` for the underlying functions, reused as-is.

In [ ]:
def pos_vs_neg_stats(category):
    wide = aggregate_by_subject_window(df[df["category"] == category], condition_col="valence")
    per_subject_bin, bins_meta = bin_by_size(wide, bin_size=BIN_SIZE, tr_offset=0)
    result = compare_conditions_by_bin(per_subject_bin, "pos", "neg", method=STAT_METHOD,
                                        alternative=POS_VS_NEG_ALTERNATIVE, bins_meta=bins_meta)
    if not result.empty:
        result.insert(0, "category", category)
    return result


pos_vs_neg = pd.concat([pos_vs_neg_stats(cat) for cat in CATEGORIES], ignore_index=True)

print("pos vs. neg:")
display(pos_vs_neg)

## Plot 2: pos vs. neg, one figure per category

Subplots per figure: each operation found above, plus "combined" (all
operations averaged together).

In [ ]:
def plot_valence_by_operation(data, title, sig_stats=None):
    per_subject = data.groupby(["subject", "operation", "valence", "time_sec"])["value"].mean().reset_index()
    combined = data.groupby(["subject", "valence", "time_sec"])["value"].mean().reset_index()
    combined["operation"] = "combined"

    panels = OPERATIONS + ["combined"]
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.5), sharex=True, sharey=True)
    for ax, panel in zip(axes, panels):
        subset = combined if panel == "combined" else per_subject[per_subject["operation"] == panel]
        sns.lineplot(data=subset, x="time_sec", y="value", hue="valence",
                     hue_order=["pos", "neg"], palette=VALENCE_COLORS, ax=ax, legend=False)
        draw_event_annotations(ax, EVENT_MARKERS, tr=TR_SECONDS, show_labels=True)
        ax.set_title(panel)
        ax.set_xlabel("Time (s)")
        if panel == "combined" and sig_stats is not None and not sig_stats.empty:
            stats_sec = sig_stats.assign(tr_start_sec=sig_stats["tr_start"] * TR_SECONDS,
                                          tr_end_sec=sig_stats["tr_end"] * TR_SECONDS)
            annotate_significance(ax, stats_sec, alpha=ALPHA, tr_start_col="tr_start_sec", tr_end_col="tr_end_sec")
        ax.set_ylabel(Y_LABEL)
    add_shared_legend(fig, VALENCE_COLORS, title="valence")
    fig.suptitle(title, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0.08, 1, 0.93])
    return fig


for cat in CATEGORIES:
    plot_valence_by_operation(df[df["category"] == cat], f"{desc}: {cat} -- pos vs. neg",
                               sig_stats=pos_vs_neg[pos_vs_neg["category"] == cat])
    plt.show()

## Save outputs (optional)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

os.makedirs(OUTPUT_DIR, exist_ok=True)
pdf_path = os.path.join(OUTPUT_DIR, f"{desc}_operation_evidence.pdf")

with PdfPages(pdf_path) as pdf:
    for valence in ("pos", "neg"):
        fig = plot_operations_by_category(df[df["valence"] == valence], f"{desc}: operation evidence by category -- {valence}")
        pdf.savefig(fig)
        plt.close(fig)
    for cat in CATEGORIES:
        fig = plot_valence_by_operation(df[df["category"] == cat], f"{desc}: {cat} -- pos vs. neg",
                                         sig_stats=pos_vs_neg[pos_vs_neg["category"] == cat])
        pdf.savefig(fig)
        plt.close(fig)

print(f"PDF written to: {pdf_path}")

pos_vs_neg.to_csv(os.path.join(OUTPUT_DIR, f"{desc}_pos_vs_neg_stats.csv"), index=False)
print(f"Stats written to: {OUTPUT_DIR}")